In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,log_loss

In [11]:
df = pd.read_csv('../0.dataset/dataset_serangan_jantung_clean.csv')
df_x = df.drop(columns='Serangan_Jantung')
df_y = df['Serangan_Jantung']

# 1.Binary Classification K-Nearest Neighbors

In [12]:
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42)

In [ ]:
params = {
 'n_neighbors' : list(range(1, 6)),
 'weights': ['uniform', 'distance'],
 'metric': ['euclidean', 'manhattan', 'minkowski'],
 'p': [1,2]
}

knn = KNeighborsClassifier()
random_search = RandomizedSearchCV(estimator=knn,param_distributions=params,
                                   n_iter=30,cv=6,scoring='accuracy',random_state=42,n_jobs=-1)
random_search.fit(X_train,y_train)
best_model_knn = random_search.best_estimator_

sfs_knn = SequentialFeatureSelector(estimator=best_model_knn,n_features_to_select=5,direction='forward')
sfs_knn.fit(X_train,y_train)

X_trains_selected = sfs_knn.transform(X_train)
X_test_selected = sfs_knn.transform(X_test)

fitur_terpilih = X_train.columns[sfs_knn.get_support()]
best_model_knn.fit(X_trains_selected,y_train)

print(f'\nHyperparameter Terbaik: {random_search.best_params_}')
print(f'\nAkurasi Validasi Terbaik: {random_search.best_score_*100:.2f}')
print(f'\nFitur Terbaik Yang Terpilih:\n {list(fitur_terpilih)}')




Hyperparameter Terbaik: {'weights': 'distance', 'p': 1, 'n_neighbors': 5, 'metric': 'minkowski'}

Akurasi Validasi Terbaik: 72.83

Fitur Terbaik Yang Terpilih:
 ['Usia', 'Tipe_Nyeri_Dada', 'Detak_Jantung_Max', 'Angina_Olahraga', 'Oldpeak_ST']
